# Experiment 3 - Observation-Frequency Distribution Shift

This notebook keeps strict validation-selected-checkpoint provenance and organizes Experiment 3 into four sub-experiments.

## Scientific Question

How much held-out performance is recovered when the training observation regime matches the deployment observation regime, and do related controls support that interpretation without changing the underlying model-selection rules?

## Experiment Definition

Sub-experiments:

- Exp3A. Structured observation-frequency shift: h=12, r={4,8}.
- Exp3B. Normalizer control: h=12, r=4.
- Exp3C. Random-matched observation-frequency shift: h=12, r=4.
- Exp3D. Longer-horizon replication: h=96, r=4.

All test predictions preserve the strict chain: validation log -> complete run -> validation-AUPRC-selected epoch -> exact selected checkpoint -> exact expected prediction filename.

In [ ]:
# Missing test predictions are always generated automatically.
# Set this to True only when intentionally regenerating ALL test predictions.
RERUN_ALL_TEST_INFERENCE = True

## Configuration

The validation log/checkpoint filename verifies only fields encoded in the run name: model seed, horizon, timestep, train sampling regime, hidden dimension, dropout/rec_dropout tokens, depth, batch size, target-replication coefficient, and L1/L2 suffixes. The notebook applies strict mismatch checks only to those filename-verified fields.

The current result tree does not include a saved per-run argument/config artifact for fields such as `network`, `optimizer`, `lr`, `beta_1`, `imputation`, or `prefix`. Those fields are therefore assumed from the known Experiment 2/3 training configuration below and are labeled as experiment-config-assumed in the provenance tables. They are still used to reconstruct test-inference commands, but they are not presented as independently verified from the training run.


In [ ]:
from pathlib import Path
import os
import re
import shlex
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from visualize_utils import (
    DEFAULT_BOOTSTRAP_SEED,
    across_seed_patient_bootstrap,
    compute_metric,
    find_selected_checkpoint,
    load_prediction_csv,
    metric_difference,
    paired_patient_bootstrap,
    paired_prediction_frame,
    prediction_metrics,
    read_validation_log,
    show_table,
    summarize_with_t_ci,
)

PROJECT_DATA_ROOT = Path(
    "/heinz-georgenas/users/mingzhul/Simultaneous-EHR/data/"
    "physionet.org/files/mimiciv/1.0/russo"
)

EXP3_REPO_DIR = Path.cwd().parent if Path.cwd().name == "visualizers" else Path.cwd()
EXP3_PYTHON = os.environ.get("EXP3_PYTHON", "python")
EXP3_DATA_DIR = PROJECT_DATA_ROOT / "data/length-of-stay"
EXP3_NORMALIZER_DIR = PROJECT_DATA_ROOT / "normalizers"
EXP3_RESULTS_DIR = PROJECT_DATA_ROOT / "results/fixed_horizon_icu_exit"
EXP3_OUTPUT_DIR = EXP3_RESULTS_DIR / "plots"
EXP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXP3_NETWORK = "mimic4models/keras_models/lstm.py"
NETWORK = EXP3_NETWORK
EXP3_HORIZONS = [12, 96]
EXP3_STRUCTURED_CONFIGS = {12: [4, 8], 96: [4]}
EXP3_RANDOM_MATCHED_CONFIGS = {12: [4]}
EXP3_NORMALIZER_CONTROL_CONFIGS = {12: [4]}
EXP3_RANDOM_MATCHED_SAMPLING_SEED = 100
EXP3_INCLUDE_OPTIONAL_RS_IF_AVAILABLE = False
EXP3_EXPECTED_MODEL_SEEDS = {0, 1, 2, 3, 4}
EXP3_EXPECTED_EPOCHS = 100
EXP3_TIMESTEP = 1.0
EXP3_REQUIRE_COMPLETE_EPOCHS = True
EXP3_ALLOW_PARTIAL_RESULTS = False
EXP3_METRICS = ["auroc", "auprc", "brier"]
EXP3_N_BOOT = int(os.environ.get("EXP3_N_BOOT", "2000"))
EXP3_BOOTSTRAP_SEED = DEFAULT_BOOTSTRAP_SEED

EXP3_FILENAME_VERIFIED_CONFIG_FIELDS = [
    "dim", "depth", "dropout", "rec_dropout", "batch_norm", "batch_size",
    "timestep", "horizon", "model_seed", "target_repl_coef", "l1", "l2",
    "train_sampling_strategy", "train_sampling_interval", "train_sampling_seed",
]
EXP3_EXPERIMENT_CONFIG_ASSUMED_FIELDS = [
    "network", "optimizer", "lr", "beta_1", "imputation", "prefix",
]
EXP3_RUN_CONFIG_ARTIFACT_PATH = None
EXP3_RUN_CONFIG_ARTIFACT_STATUS = "not found; non-filename fields use known Experiment 2/3 configuration"

EXP3_EXPECTED_CONFIG = {
    "network": EXP3_NETWORK,
    "dim": 16,
    "depth": 2,
    "dropout": 0.3,
    "rec_dropout": 0.0,
    "batch_norm": False,
    "batch_size": 8,
    "timestep": EXP3_TIMESTEP,
    "target_repl_coef": 0.0,
    "l1": 0.0,
    "l2": 0.0,
    "optimizer": "adam",
    "lr": 0.001,
    "beta_1": 0.9,
    "imputation": "previous",
    "prefix": "",
}

print("Experiment 3 horizons:", EXP3_HORIZONS)
print("Structured configs:", EXP3_STRUCTURED_CONFIGS)
print("Random-matched configs:", EXP3_RANDOM_MATCHED_CONFIGS)
print("Normalizer-control configs:", EXP3_NORMALIZER_CONTROL_CONFIGS)
print("Random-matched sampling seed:", EXP3_RANDOM_MATCHED_SAMPLING_SEED)
print("Expected model seeds:", sorted(EXP3_EXPECTED_MODEL_SEEDS))
print("Bootstrap draws:", EXP3_N_BOOT)
print("Partial final summaries allowed:", EXP3_ALLOW_PARTIAL_RESULTS)
print("Missing test predictions: AUTO-RUN")
print("Rerun all existing test predictions:", RERUN_ALL_TEST_INFERENCE)
if RERUN_ALL_TEST_INFERENCE:
    print("WARNING: ALL eligible test predictions will be regenerated.")
print("Filename-verified config fields:", EXP3_FILENAME_VERIFIED_CONFIG_FIELDS)
print("Experiment-config-assumed fields:", EXP3_EXPERIMENT_CONFIG_ASSUMED_FIELDS)
print("Run/config artifact status:", EXP3_RUN_CONFIG_ARTIFACT_STATUS)

## Run discovery and provenance

The discovery code follows the old notebook provenance chain:

`validation log -> validation-AUPRC-selected epoch -> exact checkpoint filename -> exact test prediction filename`

Matched-regime predictions must use the original filename `<checkpoint>.csv`. Shifted predictions must use an explicit `.testsample-...` suffix. The test regime is never inferred from directory name alone, and file modification time is never used.

In [ ]:
def _search(pattern, text, cast=None, default=None):
    m = re.search(pattern, text)
    if m is None:
        return default
    value = m.group(1)
    return cast(value) if cast is not None else value


def parse_sampling_from_name(name):
    sample = re.search(
        r"\.sample(?P<strategy>structured|random_matched)"
        r"\.r(?P<interval>\d+)"
        r"(?:\.sseed(?P<sseed>\d+))?",
        name,
    )
    if sample is None:
        return "none", None, None
    strategy = sample.group("strategy")
    interval = int(sample.group("interval"))
    seed = sample.group("sseed")
    return strategy, interval, int(seed) if seed is not None else None


def parse_exp3_log_name(log_path, normalizer_regime="train_matched"):
    name = log_path.name
    strategy, interval, sampling_seed = parse_sampling_from_name(name)
    l1 = _search(r"\.L1([0-9.eE+-]+)(?=\.|$)", name, float, 0.0)
    l2 = _search(r"\.L2([0-9.eE+-]+)(?=\.|$)", name, float, 0.0)
    return {
        "filename": name,
        "horizon": _search(r"\.h(\d+)", name, int),
        "timestep": _search(r"\.ts([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float),
        "model_seed": _search(r"\.seed(\d+)(?:\.|$)", name, int),
        "train_sampling_strategy": strategy,
        "train_sampling_interval": interval,
        "train_sampling_seed": sampling_seed,
        "normalizer_regime": normalizer_regime,
        "dim": _search(r"\.n(\d+)", name, int),
        "dropout": _search(r"\.d([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float, 0.0),
        "rec_dropout": _search(r"\.rd([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float, 0.0),
        "batch_norm": re.search(r"\.bn(?=\.|$)", name) is not None,
        "depth": _search(r"\.dep(\d+)", name, int),
        "batch_size": _search(r"\.bs(\d+)", name, int),
        "target_repl_coef": _search(r"\.trc([0-9]+(?:\.[0-9]+)?)(?=\.|$)", name, float, 0.0),
        "l1": l1,
        "l2": l2,
        "has_l1": l1 > 0,
        "has_l2": l2 > 0,
        "log_path": str(log_path),
        "output_dir": str(log_path.parent.parent),
    }


def effective_sampling_regime(strategy, interval, seed):
    if strategy == "none":
        return ("none",)
    if strategy == "structured":
        return ("structured", int(interval))
    if strategy == "random_matched":
        return ("random_matched", int(interval), seed)
    raise ValueError("Unknown sampling strategy {}".format(strategy))


def regimes_match(train_strategy, train_interval, train_seed, test_strategy, test_interval, test_seed):
    return effective_sampling_regime(train_strategy, train_interval, train_seed) == effective_sampling_regime(
        test_strategy, test_interval, test_seed)


def sampling_label(strategy, interval, seed=None):
    if strategy == "none":
        return "dense"
    if strategy == "structured":
        return "structured-r{}".format(int(interval))
    if strategy == "random_matched":
        return "random-r{}".format(int(interval))
    raise ValueError("Unknown sampling strategy {}".format(strategy))


def regime_label(train_strategy, train_interval, test_strategy, test_interval):
    return "{}->{}".format(
        sampling_label(train_strategy, train_interval),
        sampling_label(test_strategy, test_interval),
    )


def test_sampling_suffix(strategy, interval, seed):
    if strategy == "none":
        return "testsample-none"
    label = "testsample-{}-r{}".format(strategy, int(interval))
    if strategy == "random_matched":
        label += "-sseed{}".format(seed)
    return label


def expected_prediction_path(output_dir, checkpoint, train_strategy, train_interval, train_seed,
                             test_strategy, test_interval, test_seed):
    pred_dir = Path(output_dir) / "test_predictions"
    base = pred_dir / checkpoint.name
    if regimes_match(train_strategy, train_interval, train_seed, test_strategy, test_interval, test_seed):
        return Path(str(base) + ".csv")
    return Path(str(base) + ".{}.csv".format(test_sampling_suffix(test_strategy, test_interval, test_seed)))


def parse_prediction_test_regime(prediction_path, checkpoint, train_strategy, train_interval, train_seed):
    name = Path(prediction_path).name
    matched_name = checkpoint.name + ".csv"
    if name == matched_name:
        return train_strategy, train_interval, train_seed, "matched-original-filename"
    prefix = checkpoint.name + ".testsample-"
    if not (name.startswith(prefix) and name.endswith(".csv")):
        raise ValueError("Prediction file does not match checkpoint naming convention: {}".format(prediction_path))
    label = name[len(prefix):-4]
    if label == "none":
        return "none", None, None, "explicit-testsample-suffix"
    m = re.match(r"(?P<strategy>structured|random_matched)-r(?P<interval>\d+)(?:-sseed(?P<seed>\d+))?$", label)
    if m is None:
        raise ValueError("Unrecognized test-sampling suffix in {}".format(prediction_path))
    strategy = m.group("strategy")
    interval = int(m.group("interval"))
    seed = m.group("seed")
    return strategy, interval, int(seed) if seed is not None else None, "explicit-testsample-suffix"


def _float_mismatch(actual, expected, tol=1e-9):
    if actual is None:
        return True
    return abs(float(actual) - float(expected)) > tol


def attach_assumed_experiment_config(run):
    run = dict(run)
    for key in EXP3_EXPERIMENT_CONFIG_ASSUMED_FIELDS:
        run[key] = EXP3_EXPECTED_CONFIG[key]
        run["{}_source".format(key)] = "experiment-config-assumed"
    for key in EXP3_FILENAME_VERIFIED_CONFIG_FIELDS:
        run["{}_source".format(key)] = "filename-verified"
    run["filename_verified_config_fields"] = ", ".join(EXP3_FILENAME_VERIFIED_CONFIG_FIELDS)
    run["experiment_config_assumed_fields"] = ", ".join(EXP3_EXPERIMENT_CONFIG_ASSUMED_FIELDS)
    run["run_config_artifact_path"] = EXP3_RUN_CONFIG_ARTIFACT_PATH
    run["run_config_artifact_status"] = EXP3_RUN_CONFIG_ARTIFACT_STATUS
    return run


def hyperparameter_mismatch_reason(run):
    reasons = []
    for key in ["dim", "depth", "batch_size"]:
        if run.get(key) != EXP3_EXPECTED_CONFIG[key]:
            reasons.append("{}={} expected {}".format(key, run.get(key), EXP3_EXPECTED_CONFIG[key]))
    for key in ["dropout", "rec_dropout", "target_repl_coef", "l1", "l2", "timestep"]:
        if _float_mismatch(run.get(key), EXP3_EXPECTED_CONFIG[key]):
            reasons.append("{}={} expected {}".format(key, run.get(key), EXP3_EXPECTED_CONFIG[key]))
    if run.get("horizon") not in EXP3_HORIZONS:
        reasons.append("horizon={} not in expected {}".format(run.get("horizon"), EXP3_HORIZONS))
    if bool(run.get("batch_norm")) != bool(EXP3_EXPECTED_CONFIG["batch_norm"]):
        reasons.append("batch_norm={} expected {}".format(run.get("batch_norm"), EXP3_EXPECTED_CONFIG["batch_norm"]))
    if run.get("model_seed") not in EXP3_EXPECTED_MODEL_SEEDS:
        reasons.append("model_seed={} not in expected {}".format(run.get("model_seed"), sorted(EXP3_EXPECTED_MODEL_SEEDS)))
    return "; ".join(reasons)


def normalizer_candidates_for_regime(strategy, interval, seed):
    if strategy == "none":
        pattern = "fixed_horizon_icu_exit_ts:{:.2f}_impute:{}_start:zero_masks:True_n:*.normalizer".format(
            EXP3_TIMESTEP, EXP3_EXPECTED_CONFIG["imputation"])
    elif strategy == "structured":
        pattern = (
            "fixed_horizon_icu_exit_sampling:structured_r:{}_ts:{:.2f}"
            "_impute:{}_start:zero_masks:True_n:*.normalizer"
        ).format(int(interval), EXP3_TIMESTEP, EXP3_EXPECTED_CONFIG["imputation"])
    elif strategy == "random_matched":
        pattern = (
            "fixed_horizon_icu_exit_sampling:random_matched_r:{}_seed:{}_ts:{:.2f}"
            "_impute:{}_start:zero_masks:True_n:*.normalizer"
        ).format(int(interval), int(seed), EXP3_TIMESTEP, EXP3_EXPECTED_CONFIG["imputation"])
    else:
        raise ValueError("Unknown normalizer strategy {}".format(strategy))
    return sorted(EXP3_NORMALIZER_DIR.glob(pattern))


def resolve_normalizer_for_regime(strategy, interval, seed):
    candidates = normalizer_candidates_for_regime(strategy, interval, seed)
    if len(candidates) != 1:
        raise RuntimeError(
            "Expected exactly one normalizer for {} but found {}: {}".format(
                effective_sampling_regime(strategy, interval, seed), len(candidates), [str(x) for x in candidates]))
    return candidates[0]


def resolve_normalizer_for_run(train_strategy, train_interval, train_seed, normalizer_regime):
    if normalizer_regime == "dense":
        return resolve_normalizer_for_regime("none", None, None)
    if normalizer_regime == "train_matched":
        return resolve_normalizer_for_regime(train_strategy, train_interval, train_seed)
    raise ValueError("Unknown normalizer_regime {}".format(normalizer_regime))


def quote_command(cmd):
    return " ".join(shlex.quote(str(x)) for x in cmd)


def prediction_identity(row):
    return (
        int(row["horizon"]),
        row["train_sampling_strategy"],
        None if pd.isnull(row["train_sampling_interval"]) else int(row["train_sampling_interval"]),
        None if pd.isnull(row["train_sampling_seed"]) else int(row["train_sampling_seed"]),
        row["test_sampling_strategy"],
        None if pd.isnull(row["test_sampling_interval"]) else int(row["test_sampling_interval"]),
        None if pd.isnull(row["test_sampling_seed"]) else int(row["test_sampling_seed"]),
        row["normalizer_regime"],
        int(row["model_seed"]),
    )


def build_inference_command_parts(row):
    normalizer_path = row.get("normalizer_path")
    if not normalizer_path:
        normalizer_path = str(resolve_normalizer_for_run(
            row["train_sampling_strategy"], row["train_sampling_interval"],
            row["train_sampling_seed"], row["normalizer_regime"]))
    cmd = [
        EXP3_PYTHON, "-m", "mimic4models.fixed_horizon_icu_exit.main",
        "--mode", "test",
        "--network", row["network"],
        "--data", str(EXP3_DATA_DIR),
        "--normalizer_dir", str(EXP3_NORMALIZER_DIR),
        "--normalizer_state", str(normalizer_path),
        "--output_dir", row["output_dir"],
        "--load_state", row["checkpoint_path"],
        "--horizon", str(int(row["horizon"])),
        "--timestep", str(row["timestep"]),
        "--seed", str(int(row["model_seed"])),
        "--dim", str(int(row["dim"])),
        "--depth", str(int(row["depth"])),
        "--dropout", str(row["dropout"]),
        "--rec_dropout", str(row["rec_dropout"]),
        "--batch_size", str(int(row["batch_size"])),
        "--target_repl_coef", str(row["target_repl_coef"]),
        "--l1", str(row["l1"]),
        "--l2", str(row["l2"]),
        "--optimizer", row["optimizer"],
        "--lr", str(row["lr"]),
        "--beta_1", str(row["beta_1"]),
        "--imputation", row["imputation"],
        "--prefix", row["prefix"],
        "--sampling_strategy", row["train_sampling_strategy"],
    ]
    if bool(row.get("batch_norm")):
        cmd += ["--batch_norm", "True"]
    if row["train_sampling_strategy"] != "none":
        cmd += ["--sampling_interval", str(int(row["train_sampling_interval"]))]
    if row["train_sampling_seed"] is not None and not pd.isnull(row["train_sampling_seed"]):
        cmd += ["--sampling_seed", str(int(row["train_sampling_seed"]))]
    if not regimes_match(row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"],
                         row["test_sampling_strategy"], row["test_sampling_interval"], row["test_sampling_seed"]):
        cmd += ["--test_sampling_strategy", row["test_sampling_strategy"]]
        if row["test_sampling_strategy"] != "none":
            cmd += ["--test_sampling_interval", str(int(row["test_sampling_interval"]))]
        if row["test_sampling_seed"] is not None and not pd.isnull(row["test_sampling_seed"]):
            cmd += ["--test_sampling_seed", str(int(row["test_sampling_seed"]))]
    return cmd


def build_missing_inference_command(row):
    return quote_command(build_inference_command_parts(row))


def run_test_inference_for_row(row):
    cmd = build_inference_command_parts(row)
    prediction_path = Path(row["expected_prediction_path"])
    if prediction_path.exists():
        if not RERUN_ALL_TEST_INFERENCE:
            return
        print("RERUN_ALL_TEST_INFERENCE=True")
        print("Regenerating existing prediction:")
        print(prediction_path)
    else:
        print("Missing test prediction; running inference:")
        print(prediction_path)
    print("  experiment/subexperiment:", row["experiment"])
    print("  horizon:", row["horizon"])
    print("  model seed:", row["model_seed"])
    print("  train sampling regime:", effective_sampling_regime(row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"]))
    print("  test sampling regime:", effective_sampling_regime(row["test_sampling_strategy"], row["test_sampling_interval"], row["test_sampling_seed"]))
    print("  normalizer regime/path:", row["normalizer_regime"], row["normalizer_path"])
    print("  selected epoch:", row["selected_val_epoch"])
    print("  checkpoint:", row["checkpoint_path"])
    print("  exact expected prediction path:", prediction_path)
    print("  command:", quote_command(cmd))
    subprocess.check_call(cmd, cwd=str(EXP3_REPO_DIR))
    if not prediction_path.exists():
        raise RuntimeError("Test inference completed but exact prediction file is missing: {}".format(prediction_path))

In [ ]:
def log_dir_for_run_kind(kind, horizon, r=None):
    if kind == "dense":
        return EXP3_RESULTS_DIR / "{}h".format(int(horizon)) / "keras_logs"
    if kind == "structured":
        return EXP3_RESULTS_DIR / "structured" / "{}h".format(int(horizon)) / "{}h".format(int(r)) / "keras_logs"
    if kind == "random_matched":
        return EXP3_RESULTS_DIR / "random_matched" / "{}h".format(int(horizon)) / "{}h".format(int(r)) / "keras_logs"
    if kind == "normalizer_control_dense":
        return EXP3_RESULTS_DIR / "structured_r4_dense_norm_control" / "{}h".format(int(horizon)) / "keras_logs"
    raise ValueError("Unknown run kind {}".format(kind))


def requested_train_sources():
    sources = []
    needed_horizons = set(EXP3_HORIZONS)
    for horizon in sorted(needed_horizons):
        sources.append({"kind": "dense", "horizon": horizon, "r": None, "normalizer_regime": "train_matched"})
    for horizon, rs in EXP3_STRUCTURED_CONFIGS.items():
        for r in rs:
            sources.append({"kind": "structured", "horizon": horizon, "r": r, "normalizer_regime": "train_matched"})
    for horizon, rs in EXP3_RANDOM_MATCHED_CONFIGS.items():
        for r in rs:
            sources.append({"kind": "random_matched", "horizon": horizon, "r": r, "normalizer_regime": "train_matched"})
    for horizon, rs in EXP3_NORMALIZER_CONTROL_CONFIGS.items():
        for r in rs:
            sources.append({"kind": "normalizer_control_dense", "horizon": horizon, "r": r, "normalizer_regime": "dense"})
    return sources


train_status_rows = []
train_excluded_rows = []
train_runs = {}

for source in requested_train_sources():
    log_dir = log_dir_for_run_kind(source["kind"], source["horizon"], source["r"])
    print("Scanning {} logs:".format(source["kind"]), log_dir)
    if not log_dir.exists():
        row = dict(source)
        row["exclude_reason"] = "missing log directory"
        row["log_path"] = str(log_dir)
        train_excluded_rows.append(row)
        continue
    for log_path in sorted(log_dir.glob("*.csv")):
        parsed = attach_assumed_experiment_config(parse_exp3_log_name(log_path, source["normalizer_regime"]))
        parsed["run_kind"] = source["kind"]
        parsed["source_r"] = source["r"]
        if parsed["horizon"] != source["horizon"]:
            continue
        train_strategy = parsed["train_sampling_strategy"]
        train_interval = parsed["train_sampling_interval"]
        train_seed = parsed["train_sampling_seed"]
        is_requested = False
        if source["kind"] == "dense":
            is_requested = train_strategy == "none" and abs(parsed["timestep"] - EXP3_TIMESTEP) <= 1e-6
        elif source["kind"] == "structured":
            is_requested = (train_strategy == "structured" and train_interval == source["r"]
                            and abs(parsed["timestep"] - EXP3_TIMESTEP) <= 1e-6)
        elif source["kind"] == "random_matched":
            is_requested = (train_strategy == "random_matched" and train_interval == source["r"]
                            and train_seed == EXP3_RANDOM_MATCHED_SAMPLING_SEED
                            and abs(parsed["timestep"] - EXP3_TIMESTEP) <= 1e-6)
        elif source["kind"] == "normalizer_control_dense":
            is_requested = (train_strategy == "structured" and train_interval == source["r"]
                            and abs(parsed["timestep"] - EXP3_TIMESTEP) <= 1e-6)
        if not is_requested:
            row = dict(parsed)
            row["exclude_reason"] = "outside requested Experiment 3 train regimes"
            train_excluded_rows.append(row)
            continue

        mismatch = hyperparameter_mismatch_reason(parsed)
        if mismatch:
            row = dict(parsed)
            row["exclude_reason"] = mismatch
            train_excluded_rows.append(row)
            continue

        val = read_validation_log(log_path, expected_epochs=EXP3_EXPECTED_EPOCHS)
        if val is None:
            row = dict(parsed)
            row["exclude_reason"] = "empty validation log"
            train_excluded_rows.append(row)
            continue
        parsed.update(val)
        parsed["selected_val_epoch"] = parsed["selected_validation_epoch"]
        parsed["selected_val_auprc"] = parsed["validation_auprc"]
        parsed["selected_val_auroc"] = parsed["validation_auroc"]
        checkpoint = find_selected_checkpoint(log_path, parsed["selected_validation_epoch"])
        parsed["checkpoint_path"] = None if checkpoint is None else str(checkpoint)
        parsed["checkpoint_name"] = None if checkpoint is None else checkpoint.name
        normalizer = resolve_normalizer_for_run(
            train_strategy, train_interval, parsed["train_sampling_seed"], parsed["normalizer_regime"])
        parsed["normalizer_path"] = str(normalizer)
        parsed["normalizer_candidates"] = [str(normalizer)]
        train_status_rows.append(dict(parsed))

        if EXP3_REQUIRE_COMPLETE_EPOCHS and not parsed["complete"]:
            row = dict(parsed)
            row["exclude_reason"] = "incomplete epoch set; require exact 0..99"
            train_excluded_rows.append(row)
            continue
        if checkpoint is None:
            row = dict(parsed)
            row["exclude_reason"] = "missing validation-selected checkpoint"
            train_excluded_rows.append(row)
            continue

        key = (
            int(parsed["horizon"]), train_strategy, train_interval, parsed["train_sampling_seed"],
            parsed["normalizer_regime"], int(parsed["model_seed"]), source["kind"])
        if key in train_runs:
            raise RuntimeError("Duplicate train-run identity {}:\n{}\n{}".format(
                key, train_runs[key]["log_path"], parsed["log_path"]))
        train_runs[key] = parsed

exp3_train_status = pd.DataFrame(train_status_rows)
exp3_train_excluded = pd.DataFrame(train_excluded_rows)
show_table("Experiment 3 train-run status", exp3_train_status, max_rows=300)
show_table("Experiment 3 excluded train logs", exp3_train_excluded, max_rows=300)

In [ ]:
def train_key(horizon, strategy, interval, seed, normalizer_regime, kind, model_seed):
    return (int(horizon), strategy, interval, seed, normalizer_regime, int(model_seed), kind)


def get_train_run(horizon, strategy, interval, seed, normalizer_regime, kind, model_seed):
    return train_runs.get(train_key(horizon, strategy, interval, seed, normalizer_regime, kind, model_seed))


def _load_prediction_into_row(row, prediction_path, checkpoint):
    parsed_test_strategy, parsed_test_interval, parsed_test_seed, name_mode = parse_prediction_test_regime(
        prediction_path, checkpoint,
        row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"])
    expected_effective = effective_sampling_regime(
        row["test_sampling_strategy"], row["test_sampling_interval"], row["test_sampling_seed"])
    parsed_effective = effective_sampling_regime(parsed_test_strategy, parsed_test_interval, parsed_test_seed)
    if parsed_effective != expected_effective:
        raise RuntimeError("Prediction test-regime suffix mismatch for {}: parsed {} expected {}".format(
            prediction_path, parsed_effective, expected_effective))
    pred_df = load_prediction_csv(prediction_path)
    metrics = prediction_metrics(pred_df)
    row.update(metrics)
    row["prediction_path"] = str(prediction_path)
    row["prediction_status"] = "found"
    row["prediction_exists"] = True
    row["prediction_name_mode"] = name_mode
    row["missing_reason"] = None


def add_planned_eval(planned_rows, missing_rows, experiment, train_run, test_strategy, test_interval,
                     test_seed, condition_role, r, contrast_family):
    row = dict(train_run)
    row.update({
        "experiment": experiment,
        "test_sampling_strategy": test_strategy,
        "test_sampling_interval": test_interval,
        "test_sampling_seed": test_seed,
        "regime": regime_label(row["train_sampling_strategy"], row["train_sampling_interval"],
                               test_strategy, test_interval),
        "condition": condition_role,
        "contrast_family": contrast_family,
        "target_r": r,
    })
    checkpoint = Path(row["checkpoint_path"])
    prediction_path = expected_prediction_path(
        row["output_dir"], checkpoint,
        row["train_sampling_strategy"], row["train_sampling_interval"], row["train_sampling_seed"],
        test_strategy, test_interval, test_seed,
    )
    row["expected_prediction_path"] = str(prediction_path)
    row["prediction_path"] = None
    row["prediction_status"] = "missing"
    row["prediction_exists"] = prediction_path.exists()
    row["prediction_name_mode"] = None
    row["missing_reason"] = None
    row["suggested_inference_command"] = build_missing_inference_command(row)
    row["prediction_key"] = prediction_identity(row)
    if prediction_path.exists():
        run_test_inference_for_row(row)
        row["prediction_exists"] = prediction_path.exists()
        _load_prediction_into_row(row, prediction_path, checkpoint)
    else:
        row["missing_reason"] = "missing exact prediction for selected checkpoint/test regime"
        run_test_inference_for_row(row)
        row["prediction_exists"] = prediction_path.exists()
        _load_prediction_into_row(row, prediction_path, checkpoint)
    planned_rows.append(row)


def maybe_add_eval(planned_rows, missing_rows, experiment, horizon, train_strategy, train_interval, train_seed,
                   normalizer_regime, kind, test_strategy, test_interval, test_seed,
                   condition_role, r, contrast_family, model_seed):
    train_run = get_train_run(horizon, train_strategy, train_interval, train_seed,
                              normalizer_regime, kind, model_seed)
    if train_run is None:
        missing_rows.append({
            "experiment": experiment,
            "horizon": horizon,
            "target_r": r,
            "model_seed": model_seed,
            "train_sampling_strategy": train_strategy,
            "train_sampling_interval": train_interval,
            "train_sampling_seed": train_seed,
            "test_sampling_strategy": test_strategy,
            "test_sampling_interval": test_interval,
            "test_sampling_seed": test_seed,
            "normalizer_regime": normalizer_regime,
            "run_kind": kind,
            "condition": condition_role,
            "missing_reason": "missing eligible validation-selected train run",
        })
        return
    add_planned_eval(planned_rows, missing_rows, experiment, train_run, test_strategy, test_interval,
                     test_seed, condition_role, r, contrast_family)


planned_rows = []
missing_planned_rows = []

for seed in sorted(EXP3_EXPECTED_MODEL_SEEDS):
    # Exp3A: current structured frequency-shift experiment, h=12, r={4,8}.
    maybe_add_eval(planned_rows, missing_planned_rows, "Exp3A Structured frequency shift", 12,
                   "none", None, None, "train_matched", "dense",
                   "none", None, None, "dense->dense", 1, "structured_information", seed)
    for r in EXP3_STRUCTURED_CONFIGS.get(12, []):
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3A Structured frequency shift", 12,
                       "none", None, None, "train_matched", "dense",
                       "structured", r, None, "dense->structured-r{}".format(r), r, "structured_shift", seed)
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3A Structured frequency shift", 12,
                       "structured", r, None, "train_matched", "structured",
                       "structured", r, None, "structured-r{}->structured-r{}".format(r, r), r, "structured_shift", seed)

    # Exp3B: normalizer control, h=12, r=4.
    for r in EXP3_NORMALIZER_CONTROL_CONFIGS.get(12, []):
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3B Normalizer control", 12,
                       "structured", r, None, "train_matched", "structured",
                       "structured", r, None, "structured-r{} train + train-matched norm".format(r), r,
                       "normalizer_control", seed)
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3B Normalizer control", 12,
                       "structured", r, None, "dense", "normalizer_control_dense",
                       "structured", r, None, "structured-r{} train + dense norm".format(r), r,
                       "normalizer_control", seed)

    # Exp3C: random-matched frequency-shift experiment, h=12, r=4.
    for r in EXP3_RANDOM_MATCHED_CONFIGS.get(12, []):
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3C Random-matched frequency shift", 12,
                       "none", None, None, "train_matched", "dense",
                       "random_matched", r, EXP3_RANDOM_MATCHED_SAMPLING_SEED,
                       "dense->random-r{}".format(r), r, "random_shift", seed)
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3C Random-matched frequency shift", 12,
                       "random_matched", r, EXP3_RANDOM_MATCHED_SAMPLING_SEED, "train_matched", "random_matched",
                       "random_matched", r, EXP3_RANDOM_MATCHED_SAMPLING_SEED,
                       "random-r{}->random-r{}".format(r, r), r, "random_shift", seed)

    # Exp3D: longer-horizon structured-shift replication, h=96, r=4.
    for r in EXP3_STRUCTURED_CONFIGS.get(96, []):
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3D Horizon replication", 96,
                       "none", None, None, "train_matched", "dense",
                       "structured", r, None, "dense->structured-r{}".format(r), r, "horizon_replication", seed)
        maybe_add_eval(planned_rows, missing_planned_rows, "Exp3D Horizon replication", 96,
                       "structured", r, None, "train_matched", "structured",
                       "structured", r, None, "structured-r{}->structured-r{}".format(r, r), r,
                       "horizon_replication", seed)

exp3_run_status = pd.DataFrame(planned_rows)
if len(exp3_run_status):
    exp3_run_status = exp3_run_status.sort_values(
        ["experiment", "horizon", "target_r", "condition", "normalizer_regime", "model_seed"]
    ).reset_index(drop=True)

exp3_missing_planned_runs = pd.DataFrame(missing_planned_rows)
missing_mask = exp3_run_status["prediction_status"] != "found" if len(exp3_run_status) else []
exp3_missing_predictions = exp3_run_status[missing_mask].copy() if len(exp3_run_status) else pd.DataFrame()
if len(exp3_missing_planned_runs):
    exp3_missing_predictions = pd.concat([exp3_missing_predictions, exp3_missing_planned_runs], ignore_index=True)

exp3_run_results = exp3_run_status[exp3_run_status["prediction_status"] == "found"].copy() if len(exp3_run_status) else pd.DataFrame()


def normalized_prediction_identity_frame(df):
    required = ["stay", "y_true", "prediction"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError("Prediction frame missing required columns for duplicate identity check: {}".format(missing))
    norm = df[required].copy()
    norm["stay"] = norm["stay"].astype(str)
    return norm.sort_values(required).reset_index(drop=True)


exp3_predictions = {}
exp3_prediction_paths = {}
if len(exp3_run_results):
    for _, row in exp3_run_results.iterrows():
        key = prediction_identity(row)
        pred_path = str(row["prediction_path"])
        pred_df = load_prediction_csv(pred_path)
        if key in exp3_predictions:
            existing_path = exp3_prediction_paths[key]
            if existing_path == pred_path:
                continue
            existing = normalized_prediction_identity_frame(exp3_predictions[key])
            new_prediction = normalized_prediction_identity_frame(pred_df)
            if not existing.equals(new_prediction):
                raise RuntimeError(
                    "Duplicate prediction identity has different prediction files.\n"
                    "prediction identity: {}\n"
                    "existing prediction path: {}\n"
                    "new prediction path: {}".format(key, existing_path, pred_path))
        else:
            exp3_predictions[key] = pred_df
            exp3_prediction_paths[key] = pred_path

provenance_cols = [
    "experiment", "condition", "contrast_family", "regime", "horizon", "target_r", "model_seed",
    "train_sampling_strategy", "train_sampling_interval", "train_sampling_seed",
    "test_sampling_strategy", "test_sampling_interval", "test_sampling_seed",
    "normalizer_regime", "run_kind",
    "dim", "depth", "dropout", "rec_dropout", "batch_norm", "batch_size", "timestep",
    "target_repl_coef", "l1", "l2",
    "network", "optimizer", "lr", "beta_1", "imputation", "prefix",
    "filename_verified_config_fields", "experiment_config_assumed_fields",
    "network_source", "optimizer_source", "lr_source", "beta_1_source", "imputation_source", "prefix_source",
    "run_config_artifact_path", "run_config_artifact_status",
    "n_epochs", "complete", "selected_val_epoch", "selected_val_auprc", "selected_val_auroc",
    "normalizer_path", "checkpoint_path", "expected_prediction_path", "prediction_path",
    "prediction_exists", "prediction_status", "prediction_name_mode",
    "number_of_stays", "number_of_patients",
]
show_table("Experiment 3 compact run-status table", exp3_run_status[[c for c in provenance_cols if c in exp3_run_status.columns]], max_rows=500)
show_table("Experiment 3 missing predictions / incomplete provenance", exp3_missing_predictions[[c for c in provenance_cols + ["missing_reason", "suggested_inference_command"] if c in exp3_missing_predictions.columns]], max_rows=500)
show_table("Experiment 3 included exact prediction rows", exp3_run_results[[c for c in provenance_cols + ["test_auroc", "test_auprc", "test_brier"] if c in exp3_run_results.columns]], max_rows=500)

if len(exp3_run_results):
    for keys, group in exp3_run_results.groupby(["experiment", "horizon", "target_r", "condition", "normalizer_regime"]):
        observed = set(group["model_seed"].astype(int).tolist())
        if observed != EXP3_EXPECTED_MODEL_SEEDS:
            print("PARTIAL SEED SET for {}: observed={} missing={} extra={}".format(
                keys, sorted(observed), sorted(EXP3_EXPECTED_MODEL_SEEDS - observed),
                sorted(observed - EXP3_EXPECTED_MODEL_SEEDS)))

## Absolute Test Performance

Absolute performance is shown by sub-experiment, horizon, intended test regime, condition, and normalizer regime. These summaries are descriptive; model selection still uses validation AUPRC only.

In [ ]:
def complete_seed_set_for_group(group, seed_col="model_seed"):
    observed = set(group[seed_col].dropna().astype(int).tolist())
    return observed == EXP3_EXPECTED_MODEL_SEEDS


def complete_seed_groups(df, group_cols, seed_col="model_seed"):
    if df is None or len(df) == 0:
        return df
    if EXP3_ALLOW_PARTIAL_RESULTS:
        print("Using partial exploratory results because EXP3_ALLOW_PARTIAL_RESULTS=True.")
        return df.copy()
    keep = []
    for keys, group in df.groupby(list(group_cols)):
        observed = set(group[seed_col].astype(int).tolist())
        if observed == EXP3_EXPECTED_MODEL_SEEDS:
            keep.append(group)
        else:
            print("Incomplete seed set {} observed={} missing={} extra={}".format(
                keys, sorted(observed), sorted(EXP3_EXPECTED_MODEL_SEEDS - observed),
                sorted(observed - EXP3_EXPECTED_MODEL_SEEDS)))
    if not keep:
        return df.iloc[0:0].copy()
    return pd.concat(keep, ignore_index=True)


absolute_group_cols = ["experiment", "horizon", "target_r", "condition", "normalizer_regime"]
exp3_absolute_summary_parts = []
for metric_col in ["test_auroc", "test_auprc", "test_brier"]:
    if len(exp3_run_results):
        metric_summary = summarize_with_t_ci(exp3_run_results, absolute_group_cols, metric_col)
        metric_summary.insert(len(absolute_group_cols), "metric", metric_col)
        exp3_absolute_summary_parts.append(metric_summary)
exp3_absolute_summary = pd.concat(exp3_absolute_summary_parts, ignore_index=True) if exp3_absolute_summary_parts else pd.DataFrame()
show_table("Experiment 3 absolute test performance mean +/- 95% Student-t CI", exp3_absolute_summary, max_rows=300)


def plot_absolute_metric(metric_col, ylabel, output_name):
    if len(exp3_run_results) == 0:
        print("Skipping {}: no exact prediction rows.".format(output_name))
        return
    plot_rows = exp3_run_results[exp3_run_results["experiment"] == "Exp3A Structured frequency shift"]
    if len(plot_rows) == 0:
        print("Skipping {}: no Exp3A rows.".format(output_name))
        return
    summary = summarize_with_t_ci(plot_rows, ["condition"], metric_col)
    order = ["dense->dense", "dense->structured-r4", "structured-r4->structured-r4",
             "dense->structured-r8", "structured-r8->structured-r8"]
    summary["order"] = summary["condition"].apply(lambda x: order.index(x) if x in order else 999)
    summary = summary.sort_values("order")
    x = np.arange(len(summary))
    y = summary["mean"].values
    yerr = summary["ci95"].values
    plt.figure(figsize=(9, 4.8))
    plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=4, color="black")
    for i, condition in enumerate(summary["condition"].tolist()):
        raw = plot_rows[plot_rows["condition"] == condition]
        plt.scatter(np.repeat(i, len(raw)), raw[metric_col].values, alpha=0.45, s=24)
    plt.xticks(x, summary["condition"].tolist(), rotation=25, ha="right")
    plt.ylabel(ylabel)
    plt.xlabel("train -> test observation regime")
    plt.title("Experiment 3A {} by train/test regime\nmean across model seeds +/- 95% Student-t CI".format(ylabel))
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    output_path = EXP3_OUTPUT_DIR / output_name
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", output_path)


plot_absolute_metric("test_auroc", "Test AUROC", "experiment3_test_auroc_by_regime_ci95.png")
plot_absolute_metric("test_auprc", "Test AUPRC", "experiment3_test_auprc_by_regime_ci95.png")

## Exp3A. Structured Frequency Shift

Does training under the same structured sparse observation regime used at deployment recover performance relative to a dense-trained model evaluated on the identical sparse observations?

The information-effect analysis is descriptive: `M(dense->dense) - M(structured-r->structured-r)`. It is not treated as an additive causal decomposition with the shift effect.

In [ ]:
def find_result_row(experiment, horizon, r, condition, normalizer_regime, seed):
    if len(exp3_run_results) == 0:
        return None
    sub = exp3_run_results[
        (exp3_run_results["experiment"] == experiment) &
        (exp3_run_results["horizon"].astype(int) == int(horizon)) &
        (exp3_run_results["target_r"].astype(int) == int(r)) &
        (exp3_run_results["condition"] == condition) &
        (exp3_run_results["normalizer_regime"] == normalizer_regime) &
        (exp3_run_results["model_seed"].astype(int) == int(seed))
    ]
    if len(sub) == 0:
        return None
    if len(sub) > 1:
        raise RuntimeError("Multiple rows for {}, h={}, r={}, condition={}, norm={}, seed={}".format(
            experiment, horizon, r, condition, normalizer_regime, seed))
    return sub.iloc[0]


information_rows = []
for r in EXP3_STRUCTURED_CONFIGS.get(12, []):
    for seed in sorted(EXP3_EXPECTED_MODEL_SEEDS):
        dense = find_result_row("Exp3A Structured frequency shift", 12, 1, "dense->dense", "train_matched", seed)
        sparse = find_result_row("Exp3A Structured frequency shift", 12, r,
                                 "structured-r{}->structured-r{}".format(r, r), "train_matched", seed)
        if dense is None or sparse is None:
            continue
        for metric in EXP3_METRICS:
            col = "test_{}".format(metric)
            left_value = dense[col]
            right_value = sparse[col]
            information_rows.append({
                "experiment": "Exp3A Structured frequency shift",
                "horizon": 12,
                "r": r,
                "model_seed": seed,
                "metric": metric,
                "left_condition": "dense->dense",
                "right_condition": "structured-r{}->structured-r{}".format(r, r),
                "left_metric": left_value,
                "right_metric": right_value,
                "difference": metric_difference(left_value, right_value, metric),
                "interpretation": "descriptive information effect, not additive decomposition",
            })
exp3_information_effects = pd.DataFrame(information_rows)
show_table("Experiment 3A seed-level descriptive information effects", exp3_information_effects, max_rows=200)

if len(exp3_information_effects):
    exp3_information_summary = summarize_with_t_ci(exp3_information_effects, ["experiment", "horizon", "r", "metric"], "difference")
else:
    exp3_information_summary = pd.DataFrame()
show_table("Experiment 3A descriptive information-effect summary", exp3_information_summary, max_rows=100)

## Exp3B-D. Controls and Replications

Exp3B asks whether the apparent benefit of sparse-regime training could be explained substantially by using different normalization statistics.

Exp3C asks whether the observation-regime mismatch effect persists when sparsity is random/count-matched rather than caused by periodic structured thinning.

Exp3D asks whether the observation-regime mismatch effect also appears for a substantially longer prediction horizon.

In [ ]:
def verify_same_test_examples(left_key, right_key):
    paired = paired_prediction_frame(exp3_predictions[left_key], exp3_predictions[right_key])
    return int(len(paired)), int(paired["patient_id"].nunique())


def contrast_seed(metric, left_row, right_row, contrast_seed_value):
    left_key = prediction_identity(left_row)
    right_key = prediction_identity(right_row)
    paired_n_stays, paired_n_patients = verify_same_test_examples(left_key, right_key)
    result = paired_patient_bootstrap(
        exp3_predictions[left_key],
        exp3_predictions[right_key],
        metric=metric,
        n_boot=EXP3_N_BOOT,
        random_state=contrast_seed_value,
    )
    return result, paired_n_stays, paired_n_patients, left_key, right_key


def compute_seed_level_contrasts(specs):
    rows = []
    for spec in specs:
        for seed in sorted(EXP3_EXPECTED_MODEL_SEEDS):
            left = find_result_row(spec["experiment"], spec["horizon"], spec["r"],
                                   spec["left_condition"], spec["left_normalizer_regime"], seed)
            right = find_result_row(spec["experiment"], spec["horizon"], spec["r"],
                                    spec["right_condition"], spec["right_normalizer_regime"], seed)
            if left is None or right is None:
                continue
            for metric in EXP3_METRICS:
                boot_seed = (EXP3_BOOTSTRAP_SEED + int(spec["horizon"]) * 10000
                             + int(spec["r"]) * 1000 + int(seed) * 10 + EXP3_METRICS.index(metric))
                result, paired_n_stays, paired_n_patients, left_key, right_key = contrast_seed(
                    metric, left, right, boot_seed)
                rows.append({
                    "experiment": spec["experiment"],
                    "contrast": spec["contrast"],
                    "horizon": spec["horizon"],
                    "r": spec["r"],
                    "model_seed": seed,
                    "metric": metric,
                    "left_condition": spec["left_condition"],
                    "right_condition": spec["right_condition"],
                    "left_normalizer_regime": spec["left_normalizer_regime"],
                    "right_normalizer_regime": spec["right_normalizer_regime"],
                    "left_prediction_key": left_key,
                    "right_prediction_key": right_key,
                    "difference": result["difference"],
                    "left_metric": result["left_metric"],
                    "right_metric": result["right_metric"],
                    "patient_bootstrap_ci_low": result["ci_low"],
                    "patient_bootstrap_ci_high": result["ci_high"],
                    "n_patients": result["n_patients"],
                    "n_stays": result["n_stays"],
                    "verified_same_test_examples": True,
                    "verified_same_n_stays": paired_n_stays,
                    "verified_same_n_patients": paired_n_patients,
                    "n_boot_valid": result["n_boot_valid"],
                })
    return pd.DataFrame(rows)


def summarize_contrasts(seed_level_df):
    summary_rows = []
    if seed_level_df is None or len(seed_level_df) == 0:
        return pd.DataFrame()
    group_cols = ["experiment", "contrast", "horizon", "r", "metric", "left_condition", "right_condition",
                  "left_normalizer_regime", "right_normalizer_regime"]
    for keys, group in seed_level_df.groupby(group_cols):
        (experiment, contrast, horizon, r, metric, left_condition, right_condition,
         left_norm, right_norm) = keys
        seeds = sorted(group["model_seed"].astype(int).unique().tolist())
        complete_seed_set = set(seeds) == EXP3_EXPECTED_MODEL_SEEDS
        if not complete_seed_set:
            print("PARTIAL CONTRAST SEED SET: {} h={} r={} metric={} observed={} missing={} extra={}".format(
                contrast, horizon, r, metric, sorted(seeds),
                sorted(EXP3_EXPECTED_MODEL_SEEDS - set(seeds)), sorted(set(seeds) - EXP3_EXPECTED_MODEL_SEEDS)))
            if not EXP3_ALLOW_PARTIAL_RESULTS:
                print("Skipping final across-seed summary for incomplete contrast because EXP3_ALLOW_PARTIAL_RESULTS=False.")
                continue
        bootstrap_keys = []
        for seed in seeds:
            one = group[group["model_seed"].astype(int) == int(seed)].iloc[0]
            bootstrap_keys.append((seed, one["left_prediction_key"], one["right_prediction_key"]))
        boot = across_seed_patient_bootstrap(
            exp3_predictions,
            bootstrap_keys,
            metric=metric,
            n_boot=EXP3_N_BOOT,
            random_state=(EXP3_BOOTSTRAP_SEED + int(horizon) * 10000 + int(r) * 1000 + EXP3_METRICS.index(metric)),
        ) if len(bootstrap_keys) else {
            "patient_bootstrap_ci_low": np.nan,
            "patient_bootstrap_ci_high": np.nan,
        }
        diffs = group["difference"].astype(float).values
        summary_rows.append({
            "experiment": experiment,
            "contrast": contrast,
            "horizon": int(horizon),
            "r": int(r),
            "metric": metric,
            "left_condition": left_condition,
            "right_condition": right_condition,
            "left_normalizer_regime": left_norm,
            "right_normalizer_regime": right_norm,
            "left_mean": float(np.mean(group["left_metric"].astype(float).values)),
            "right_mean": float(np.mean(group["right_metric"].astype(float).values)),
            "mean_difference": float(np.mean(diffs)) if len(diffs) else np.nan,
            "sd_difference": float(np.std(diffs, ddof=1)) if len(diffs) > 1 else np.nan,
            "n_seeds": int(len(seeds)),
            "expected_seeds": int(len(EXP3_EXPECTED_MODEL_SEEDS)),
            "complete_seed_set": complete_seed_set,
            "patient_bootstrap_ci_low": boot["patient_bootstrap_ci_low"],
            "patient_bootstrap_ci_high": boot["patient_bootstrap_ci_high"],
        })
    return pd.DataFrame(summary_rows)


structured_specs = []
for r in EXP3_STRUCTURED_CONFIGS.get(12, []):
    structured_specs.append({
        "experiment": "Exp3A Structured frequency shift",
        "contrast": "structured shift",
        "horizon": 12,
        "r": r,
        "left_condition": "structured-r{}->structured-r{}".format(r, r),
        "right_condition": "dense->structured-r{}".format(r),
        "left_normalizer_regime": "train_matched",
        "right_normalizer_regime": "train_matched",
    })

normalizer_specs = [{
    "experiment": "Exp3B Normalizer control",
    "contrast": "normalizer control",
    "horizon": 12,
    "r": 4,
    "left_condition": "structured-r4 train + train-matched norm",
    "right_condition": "structured-r4 train + dense norm",
    "left_normalizer_regime": "train_matched",
    "right_normalizer_regime": "dense",
}]

random_specs = [{
    "experiment": "Exp3C Random-matched frequency shift",
    "contrast": "random-matched shift",
    "horizon": 12,
    "r": 4,
    "left_condition": "random-r4->random-r4",
    "right_condition": "dense->random-r4",
    "left_normalizer_regime": "train_matched",
    "right_normalizer_regime": "train_matched",
}]

horizon_specs = [{
    "experiment": "Exp3D Horizon replication",
    "contrast": "structured shift",
    "horizon": 96,
    "r": 4,
    "left_condition": "structured-r4->structured-r4",
    "right_condition": "dense->structured-r4",
    "left_normalizer_regime": "train_matched",
    "right_normalizer_regime": "train_matched",
}]

exp3_structured_shift_contrasts = compute_seed_level_contrasts(structured_specs)
exp3_normalizer_control_contrasts = compute_seed_level_contrasts(normalizer_specs)
exp3_random_shift_contrasts = compute_seed_level_contrasts(random_specs)
exp3_horizon_replication_contrasts = compute_seed_level_contrasts(horizon_specs)

show_table("Exp3A. Structured frequency shift seed-level paired-bootstrap contrasts", exp3_structured_shift_contrasts, max_rows=300)
show_table("Exp3B. Normalizer control seed-level paired-bootstrap contrasts", exp3_normalizer_control_contrasts, max_rows=300)
show_table("Exp3C. Random-matched frequency shift seed-level paired-bootstrap contrasts", exp3_random_shift_contrasts, max_rows=300)
show_table("Exp3D. Horizon replication seed-level paired-bootstrap contrasts", exp3_horizon_replication_contrasts, max_rows=300)

exp3_structured_shift_summary = summarize_contrasts(exp3_structured_shift_contrasts)
exp3_normalizer_control_summary = summarize_contrasts(exp3_normalizer_control_contrasts)
exp3_random_shift_summary = summarize_contrasts(exp3_random_shift_contrasts)
exp3_horizon_replication_summary = summarize_contrasts(exp3_horizon_replication_contrasts)

show_table("Exp3A. Structured frequency shift across-seed summary", exp3_structured_shift_summary, max_rows=100)
show_table("Exp3B. Normalizer control across-seed summary", exp3_normalizer_control_summary, max_rows=100)
show_table("Exp3C. Random-matched frequency shift across-seed summary", exp3_random_shift_summary, max_rows=100)
show_table("Exp3D. Horizon replication across-seed summary", exp3_horizon_replication_summary, max_rows=100)

# Backward-compatible aliases for the original h=12 structured shift outputs.
exp3_shift_contrasts = exp3_structured_shift_contrasts
exp3_across_seed_summary = exp3_structured_shift_summary
exp3_shift_summary = exp3_structured_shift_summary

In [ ]:
def plot_shift_penalty(metric, ylabel, output_name):
    if len(exp3_structured_shift_summary) == 0:
        print("Skipping {}: no structured-shift summaries.".format(output_name))
        return
    sub = exp3_structured_shift_summary[exp3_structured_shift_summary["metric"] == metric].sort_values("r")
    if len(sub) == 0:
        print("Skipping {}: no rows for metric {}.".format(output_name, metric))
        return
    x = np.arange(len(sub))
    y = sub["mean_difference"].values
    lo = sub["patient_bootstrap_ci_low"].values
    hi = sub["patient_bootstrap_ci_high"].values
    yerr = np.vstack([y - lo, hi - y])
    plt.figure(figsize=(6.5, 4.5))
    plt.axhline(0, color="gray", linewidth=1, linestyle="--")
    plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=5, color="black")
    for i, r in enumerate(sub["r"].tolist()):
        raw = exp3_structured_shift_contrasts[
            (exp3_structured_shift_contrasts["metric"] == metric) &
            (exp3_structured_shift_contrasts["r"].astype(int) == int(r))]
        plt.scatter(np.repeat(i, len(raw)), raw["difference"].values, alpha=0.45, s=24)
    plt.xticks(x, ["r={}".format(int(r)) for r in sub["r"].tolist()])
    plt.xlabel("target structured test frequency")
    plt.ylabel(ylabel)
    plt.title("Experiment 3A {} shift penalty\nM(structured-r->structured-r) - M(dense->structured-r)".format(metric.upper()))
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    output_path = EXP3_OUTPUT_DIR / output_name
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", output_path)


plot_shift_penalty("auroc", "AUROC shift penalty (left better +)", "experiment3_shift_penalty_auroc_bootstrap_ci95.png")
plot_shift_penalty("auprc", "AUPRC shift penalty (left better +)", "experiment3_shift_penalty_auprc_bootstrap_ci95.png")

In [ ]:
summary_frames = [
    exp3_structured_shift_summary,
    exp3_normalizer_control_summary,
    exp3_random_shift_summary,
    exp3_horizon_replication_summary,
]
summary_frames = [df for df in summary_frames if df is not None and len(df)]
exp3_combined_key_contrast_summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
if len(exp3_combined_key_contrast_summary):
    final_key_cols = ["experiment", "horizon", "r", "contrast", "metric"]
    dup = exp3_combined_key_contrast_summary[
        exp3_combined_key_contrast_summary.duplicated(final_key_cols, keep=False)
    ]
    if len(dup):
        raise RuntimeError("Duplicate final contrast rows found:\n{}".format(
            dup[final_key_cols + ["left_condition", "right_condition"]]))
    inconsistent = exp3_combined_key_contrast_summary[
        (exp3_combined_key_contrast_summary["complete_seed_set"] == True) &
        (exp3_combined_key_contrast_summary["n_seeds"].astype(int) != len(EXP3_EXPECTED_MODEL_SEEDS))
    ]
    if len(inconsistent):
        raise RuntimeError("Final contrast row marked complete with n_seeds != {}:\n{}".format(
            len(EXP3_EXPECTED_MODEL_SEEDS), inconsistent[final_key_cols + ["n_seeds", "complete_seed_set"]]))
    exp3_combined_key_contrast_summary = exp3_combined_key_contrast_summary.sort_values(
        ["experiment", "horizon", "r", "metric"]
    ).reset_index(drop=True)

combined_cols = [
    "experiment", "horizon", "r", "metric", "left_condition", "right_condition",
    "left_mean", "right_mean", "mean_difference", "sd_difference", "n_seeds",
    "complete_seed_set", "patient_bootstrap_ci_low", "patient_bootstrap_ci_high",
]
show_table(
    "Experiment 3 combined key-contrast summary",
    exp3_combined_key_contrast_summary[[c for c in combined_cols if c in exp3_combined_key_contrast_summary.columns]],
    max_rows=300,
)

expected_final_contrasts = [
    ("Exp3A Structured frequency shift", 12, 4, "structured shift"),
    ("Exp3A Structured frequency shift", 12, 8, "structured shift"),
    ("Exp3B Normalizer control", 12, 4, "normalizer control"),
    ("Exp3C Random-matched frequency shift", 12, 4, "random-matched shift"),
    ("Exp3D Horizon replication", 96, 4, "structured shift"),
]
validation_rows = []
for experiment, horizon, r, contrast in expected_final_contrasts:
    for metric in EXP3_METRICS:
        present = False
        n_seeds = 0
        complete_seed_set = False
        if len(exp3_combined_key_contrast_summary):
            matches = exp3_combined_key_contrast_summary[
                (exp3_combined_key_contrast_summary["experiment"] == experiment) &
                (exp3_combined_key_contrast_summary["horizon"].astype(int) == int(horizon)) &
                (exp3_combined_key_contrast_summary["r"].astype(int) == int(r)) &
                (exp3_combined_key_contrast_summary["contrast"] == contrast) &
                (exp3_combined_key_contrast_summary["metric"] == metric)
            ]
            if len(matches) > 1:
                raise RuntimeError("Duplicate final rows for {} h={} r={} contrast={} metric={}".format(
                    experiment, horizon, r, contrast, metric))
            if len(matches) == 1:
                present = True
                n_seeds = int(matches.iloc[0]["n_seeds"])
                complete_seed_set = bool(matches.iloc[0]["complete_seed_set"])
                if complete_seed_set and n_seeds != len(EXP3_EXPECTED_MODEL_SEEDS):
                    raise RuntimeError("Final row marked complete with n_seeds={} for {} h={} r={} metric={}".format(
                        n_seeds, experiment, horizon, r, metric))
        if not present or not complete_seed_set:
            print("INCOMPLETE FINAL CONTRAST: {} h={} r={} {}".format(
                experiment, horizon, r, metric.upper()))
        validation_rows.append({
            "experiment": experiment,
            "horizon": horizon,
            "r": r,
            "contrast": contrast,
            "metric": metric,
            "present_in_final_summary": present,
            "n_seeds": n_seeds,
            "complete_seed_set": complete_seed_set,
        })

exp3_final_contrast_coverage = pd.DataFrame(validation_rows)
show_table("Experiment 3 final contrast coverage", exp3_final_contrast_coverage, max_rows=100)

## Across-Seed Summary

The combined key-contrast table above is the compact cross-experiment summary. Structured-shift information effects remain separate because they are descriptive rather than an additive decomposition.

## Go/No-Go Interpretation

Interpret contrasts only where the intended validation-selected checkpoints and exact test-prediction files are present. Positive differences mean the left condition performs better; Brier uses the same left-better sign convention by reversing the raw lower-is-better metric.

## Source-Notebook Mapping

Exp3A preserves the original structured h=12 analysis. Exp3B adds the dense-normalizer control, Exp3C adds random-matched sampling, and Exp3D adds the h=96 structured-shift replication. All sections use the same strict provenance, exact prediction naming, automatic missing-prediction inference, same-test-example checks, and paired patient bootstrap helpers.